In [ ]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
load_dotenv()
from typing import Literal
from pydantic import Field,BaseModel


In [60]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0,
    base_url="http://172.31.0.1:11434"
)


In [61]:
class Sentiment(BaseModel):
    sentiment:Literal["positive", "negative", "neutral"]=Field(description="The sentiment of the review")
    

In [62]:
struct_model=llm.with_structured_output(Sentiment)

In [63]:
prompt="what is the sentiment of the review: 'The product is great!'?"
result=struct_model.invoke(prompt)

In [64]:
print(result.sentiment)

positive


In [65]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["positive", "negative", "neutral"]
    diagnosis:dict
    response:str

In [66]:
def find_sentiment(state:ReviewState):
    prompt=f"what is the sentiment of the review: '{state['review']}'?"
    result=struct_model.invoke(prompt).sentiment
    return {'sentiment':result}

In [71]:
def check_sentiment(state:ReviewState)->Literal["positive_response","run_diagnosis"]:
    if state['sentiment']=="positive":
        return "positive_response"
    else:
        return "run_diagnosis"

In [72]:
def positive_response(state:ReviewState):
    return {'response':"Thank you for your positive review!"}
def run_diagnosis(state:ReviewState):
    prompt=f"Diagnose the issue in the review: '{state['review']}'?"
    result=llm.invoke(prompt).content
    return {'diagnosis':result}
def negative_response(state:ReviewState):
    return {'response':"We are sorry to hear about your experience. We will look into the issue."}

In [73]:
graph=StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis',END)
graph.add_edge('negative_response',END)

workflow=graph.compile()

In [74]:
initial_state={'review':"The product is too bad"}
final_state=workflow.invoke(initial_state)
print(final_state)

{'review': 'The product is too bad', 'sentiment': 'negative', 'diagnosis': 'A very concise and blunt review!\n\nIn this case, I would diagnose the issue as a lack of specificity. The reviewer has clearly stated their negative opinion about the product, but they haven\'t provided any concrete reasons why it\'s "too bad". This makes it difficult for others to understand what exactly went wrong with the product or how it failed to meet expectations.\n\nA more effective review might provide specific details, such as:\n\n* What specifically didn\'t work as expected (e.g. quality issues, performance problems, etc.)\n* How it affected their experience or usage\n* Any relevant context or background information\n\nBy providing more context and specifics, the reviewer can help others make a more informed decision about whether to purchase the product themselves.'}
